In [11]:
import cv2
import json
import numpy as np
import glob
import os

curfolder = os.getcwd()
# List folders in a main folder
folderstotrack = glob.glob(os.path.join(curfolder, 'projectdata', '*'))

# Get all folders per participant, per session
pcnfolders = []

for i in folderstotrack:
    pcnfolders_in_session = glob.glob(os.path.join(i, '*'))
    pcnfolders = pcnfolders + pcnfolders_in_session

# There might be some other things we don't want now
pcnfolders = [x for x in pcnfolders if 'Config' not in x]
pcnfolders = [x for x in pcnfolders if 'opensim' not in x]
pcnfolders = [x for x in pcnfolders if 'xml' not in x]
pcnfolders = [x for x in pcnfolders if 'ResultsInverseDynamics' not in x]
pcnfolders = [x for x in pcnfolders if 'ResultsInverseKinematics' not in x]
pcnfolders = [x for x in pcnfolders if 'sto' not in x]
pcnfolders = [x for x in pcnfolders if 'txt' not in x]
pcnfolders = [x for x in pcnfolders if 'calibration' not in x]

print(pcnfolders[0:10])

['f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_processing\\projectdata\\Session_10_1\\10_1_11_p1', 'f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_processing\\projectdata\\Session_10_1\\10_1_12_p1', 'f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_processing\\projectdata\\Session_10_1\\10_1_13_p1', 'f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_processing\\projectdata\\Session_10_1\\10_1_14_p1', 'f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_processing\\projectdata\\Session_10_1\\10_1_15_p1', 'f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_processing\\projectdata\\Session_10_1\\10_1_16_p1', 'f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_processing\\projectdata\\Session_10_1\\10_1_17_p1', 'f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_processing\\projectdata\\Session_10_1\\10_1_20_p0', 'f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_processing\\projectdata\\Session_10_1\\10_1_21_p0', 'f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_p

In [12]:

# ---------------------------------------------------------------------------
# Masking
# ---------------------------------------------------------------------------

def get_fullbody_bbox(person, frame_shape, padding=1.15, conf_thresh=0.1):
    """
    Bounding box around all confidently-detected keypoints for one person,
    pooling body + hand + face arrays (the OpenPose "135-keypoint" output
    produced when running with --face --hand on top of BODY_25).

    `person` is one entry from the JSON "people" list. Reading every
    available array (rather than just pose_keypoints_2d) matters here:
    an outstretched hand or a head turned past the shoulder line can sit
    outside the body-joint-only box, which would leave a sliver unmasked.

    padding expands the box slightly so masking covers e.g. hair/fingertips
    right at the edges.
    Returns (x1, y1, x2, y2) or None if too few confident points.
    """
    keys = [
        'pose_keypoints_2d',
        'face_keypoints_2d',
        'hand_left_keypoints_2d',
        'hand_right_keypoints_2d',
    ]

    all_points = []
    for key in keys:
        arr = person.get(key)
        if not arr:
            continue
        kp = np.array(arr).reshape(-1, 3)  # x, y, confidence
        all_points.append(kp)

    if not all_points:
        return None

    kp = np.vstack(all_points)
    valid = kp[kp[:, 2] > conf_thresh]

    if len(valid) < 3:  # too few points to trust a box
        return None

    xs, ys = valid[:, 0], valid[:, 1]
    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()

    # Expand box outward from its center by `padding`
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    half_w = (x2 - x1) / 2 * padding
    half_h = (y2 - y1) / 2 * padding

    h, w = frame_shape[:2]
    x1 = int(max(cx - half_w, 0))
    x2 = int(min(cx + half_w, w))
    y1 = int(max(cy - half_h, 0))
    y2 = int(min(cy + half_h, h))

    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2, y2


def mask_video_with_keypoints(video_path, json_folder, output_path, mode='blur', fourcc_str='XVID'):
    """
    mode: 'blur' (Gaussian blur — retains rough silhouette/pose context)
          'black' (solid fill — full de-id, no shape info retained)
    Returns output_path on success, None if the video couldn't be opened.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f'  Could not open video: {video_path}')
        return None

    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*fourcc_str)
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    json_files = sorted(glob.glob(os.path.join(json_folder, '*_keypoints.json')))
    n_json = len(json_files)

    frame_idx = 0
    n_frames_masked = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx < n_json:
            with open(json_files[frame_idx], 'r') as f:
                data = json.load(f)

            for person in data.get('people', []):
                bbox = get_fullbody_bbox(person, frame.shape)
                if bbox is None:
                    continue
                x1, y1, x2, y2 = bbox
                n_frames_masked += 1

                if mode == 'blur':
                    region = frame[y1:y2, x1:x2]
                    if region.size > 0:
                        k = max(31, (min(x2 - x1, y2 - y1) // 2) * 2 + 1)
                        frame[y1:y2, x1:x2] = cv2.GaussianBlur(region, (k, k), 0)
                elif mode == 'black':
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 0), -1)

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()

    if frame_idx > n_json:
        print(f'  Note: video had {frame_idx} frames but only {n_json} JSON files '
              f'— frames {n_json}-{frame_idx-1} were written unmasked.')

    print(f'  Masked {video_path} -> {output_path} ({frame_idx} frames)')
    return output_path


# ---------------------------------------------------------------------------
# Concatenation
# ---------------------------------------------------------------------------

def concat_videos_horizontally(video_paths, output_path, target_height=None, fourcc_str='XVID'):
    """
    Stack 2+ videos side by side into one video. Videos may have different
    native resolutions/frame counts; each is resized to a common height
    (preserving aspect ratio) and the combined video runs only as long as
    the shortest input.
    """
    caps = [cv2.VideoCapture(p) for p in video_paths]
    for p, c in zip(video_paths, caps):
        if not c.isOpened():
            raise RuntimeError(f'Could not open {p} for concatenation')

    fps_list = [c.get(cv2.CAP_PROP_FPS) or 25.0 for c in caps]
    fps = min(fps_list)  # safest common denominator
    if len(set(round(f, 2) for f in fps_list)) > 1:
        print(f'  Note: input fps differ {fps_list} — using {fps} for output.')

    heights = [int(c.get(cv2.CAP_PROP_FRAME_HEIGHT)) for c in caps]
    widths = [int(c.get(cv2.CAP_PROP_FRAME_WIDTH)) for c in caps]

    if target_height is None:
        target_height = min(heights)  # avoid upscaling

    target_widths = [int(w * (target_height / h)) for w, h in zip(widths, heights)]
    combined_width = sum(target_widths)

    fourcc = cv2.VideoWriter_fourcc(*fourcc_str)
    out = cv2.VideoWriter(output_path, fourcc, fps, (combined_width, target_height))

    frame_count = 0
    while True:
        frames = []
        all_ok = True
        for c, tw in zip(caps, target_widths):
            ret, frame = c.read()
            if not ret:
                all_ok = False
                break
            frame = cv2.resize(frame, (tw, target_height))
            frames.append(frame)

        if not all_ok:
            break  # stop at the shortest video

        combined = np.hstack(frames)
        out.write(combined)
        frame_count += 1

    for c in caps:
        c.release()
    out.release()

    print(f'  Concatenated {len(video_paths)} videos -> {output_path} ({frame_count} frames)')


# ---------------------------------------------------------------------------
# Main driver
# ---------------------------------------------------------------------------

def process_session(session_folder, mask_mode='blur', cleanup_temp=False):
    """
    session_folder: path to one participant/session folder, expected to contain
        pose-2d-trackingvideos/*.avi   (3 camera videos)
        pose/pose_cam1_json, pose_cam2_json, pose_cam3_json
    Writes:
        pose-2d-masked/maskedN.avi     (per-camera masked videos)
        pose-2d-masked/combined.avi    (3 views side by side)
    """
    print(f'Processing {session_folder}')

    tracking_videos = sorted(glob.glob(os.path.join(session_folder, 'pose-2d-trackingvideos', '*.avi')))
    if len(tracking_videos) != 3:
        print(f'  Expected 3 tracking videos, found {len(tracking_videos)} — skipping.')
        return

    json_folders = [
        os.path.join(session_folder, 'pose', 'pose_cam1_json'),
        os.path.join(session_folder, 'pose', 'pose_cam2_json'),
        os.path.join(session_folder, 'pose', 'pose_cam3_json'),
    ]

    masked_folder = os.path.join(session_folder, 'pose-2d-masked')
    os.makedirs(masked_folder, exist_ok=True)

    combined_path = os.path.join(masked_folder, 'combined.avi')
    if os.path.exists(combined_path):
        print('  combined.avi already exists — skipping session.')
        return

    masked_paths = []
    for it, (video_path, json_folder) in enumerate(zip(tracking_videos, json_folders)):
        out_path = os.path.join(masked_folder, f'masked{it}.avi')
        result = mask_video_with_keypoints(video_path, json_folder, out_path, mode=mask_mode)
        if result:
            masked_paths.append(result)

    if len(masked_paths) != 3:
        print('  Not all three cameras masked successfully — skipping concatenation.')
        return

    concat_videos_horizontally(masked_paths, combined_path)

    if cleanup_temp:
        for p in masked_paths:
            os.remove(p)



In [ ]:
# --- Edit these for your setup ---
#pcnfolders = []        # list of session folder paths, same as your main pipeline
mask_mode = 'black'    # 'blur' or 'black' — full de-id, so 'black' is the safer default
cleanup_temp = False   # set True to delete the per-camera masked videos, keep only combined.avi

for session in pcnfolders:
    print(session)
    process_session(session, mask_mode=mask_mode, cleanup_temp=cleanup_temp)
    break

f:\FLESH_ContinuousBodilyEffort\02_MotionTracking_processing\projectdata\Session_10_1\10_1_11_p1
Processing f:\FLESH_ContinuousBodilyEffort\02_MotionTracking_processing\projectdata\Session_10_1\10_1_11_p1
  Masked f:\FLESH_ContinuousBodilyEffort\02_MotionTracking_processing\projectdata\Session_10_1\10_1_11_p1\pose-2d-trackingvideos\video0.avi -> f:\FLESH_ContinuousBodilyEffort\02_MotionTracking_processing\projectdata\Session_10_1\10_1_11_p1\pose-2d-masked\masked0.avi (557 frames)
